In [3]:
#!pip install mediapipe
#pip install opencv-python

In [7]:
import cv2
import numpy as np
import mediapipe as mp
import time

In [10]:
# Accessing Camera using  Open CV
cap = cv2.VideoCapture(0)
cap.set(3, 1280)
cap.set(4, 720)

# Recognize hand using MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
mp_draw = mp.solutions.drawing_utils

#Canvas
canvas = np.zeros((720, 1280, 3), dtype=np.uint8)

#Drawing Variables
prev_x, prev_y = 0, 0
brush_thickness = 8
eraser_thickness = 60
draw_color = (0, 0, 255)

#Colour and Eraser Labels 
header = np.zeros((100, 1280, 3), dtype=np.uint8)

colors = [
    ((0, 0, 255), "RED"),
    ((255, 0, 0), "BLUE"),
    ((0, 255, 0), "GREEN"),
    ((0, 255, 255), "YELLOW")
]


save_count = 1
save_time = 0
show_save_msg = False
show_empty_msg = False
has_drawing = False

# Finger Detection
def fingers_up(lm):
    tips = [4, 8, 12, 16, 20]
    fingers = []
    fingers.append(lm[tips[0]].x < lm[tips[0] - 1].x)
    for i in range(1, 5):
        fingers.append(lm[tips[i]].y < lm[tips[i] - 2].y)
    return fingers


while True:
    success, img = cap.read()
    if not success:
        break

    img = cv2.flip(img, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    #shape of labels and its texts
    header[:] = (50, 50, 50)
    for i, (color, name) in enumerate(colors):
        cv2.rectangle(header, (i * 200, 0), ((i + 1) * 200, 100), color, -1)
        cv2.putText(header, name, (i * 200 + 40, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.rectangle(header, (800, 0), (1280, 100), (0, 0, 0), -1)
    cv2.putText(header, "ERASER", (930, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    if results.multi_hand_landmarks:
        for handLms in results.multi_hand_landmarks:
            lm = handLms.landmark
            finger_status = fingers_up(lm)

            x = int(lm[8].x * 1280)
            y = int(lm[8].y * 720)

            # Colour Selection Mode 
            if finger_status[1] and finger_status[2]:
                prev_x, prev_y = 0, 0
                if y < 100:
                    if 0 < x < 200:
                        draw_color = colors[0][0]
                    elif 200 < x < 400:
                        draw_color = colors[1][0]
                    elif 400 < x < 600:
                        draw_color = colors[2][0]
                    elif 600 < x < 800:
                        draw_color = colors[3][0]
                    elif 800 < x < 1280:
                        draw_color = (0, 0, 0)

                cv2.rectangle(img, (x - 20, y - 20),
                              (x + 20, y + 20), draw_color, 2)

            #Drawing Mode
            elif finger_status[1] and not finger_status[2]:
                if prev_x == 0 and prev_y == 0:
                    prev_x, prev_y = x, y

                thickness = eraser_thickness if draw_color == (0, 0, 0) else brush_thickness
                cv2.line(canvas, (prev_x, prev_y), (x, y), draw_color, thickness)

                has_drawing = True  # ✅ drawing detected
                prev_x, prev_y = x, y

            else:
                prev_x, prev_y = 0, 0

            mp_draw.draw_landmarks(img, handLms, mp_hands.HAND_CONNECTIONS)


    gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    _, inv = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY_INV)
    inv = cv2.cvtColor(inv, cv2.COLOR_GRAY2BGR)

    img = cv2.bitwise_and(img, inv)
    img = cv2.bitwise_or(img, canvas)

    img[:100, :] = header

    # Notification messages while Saving
    if show_save_msg:
        cv2.putText(img, "Drawing Saved!",
                    (900, 680), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 0), 2)
        if time.time() - save_time > 1.5:
            show_save_msg = False

    if show_empty_msg:
        cv2.putText(img, "No drawing found!",
                    (850, 680), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 0, 255), 2)
        if time.time() - save_time > 1.5:
            show_empty_msg = False

    cv2.imshow("Virtual Painter", img)

    #Saving the drawing  and Quitting
    key = cv2.waitKey(1) & 0xFF

    
    if key == ord('s'):
        save_time = time.time()
        if has_drawing:
            filename = f"drawing_{save_count}.png"
            cv2.imwrite(filename, canvas)
            print(f"Image saved as: {filename}")
            save_count += 1
            show_save_msg = True
            show_empty_msg = False
        else:
            show_empty_msg = True
            show_save_msg = False

    
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()